In [25]:
#!pip install pandas numpy matplotlib missingno

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import missingno as msno

import os
from pathlib import Path

#import sklearn.preprocessing

pd.options.display.max_rows=1000

import sys


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the parent directory containing src/data.py."""
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate

    raise FileNotFoundError("Could not find src/data.py")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import load_data, clean_prices, clean_stock_list, clean_financials
from config import ProjectConfig, ProjectPaths

config = ProjectConfig()
paths = ProjectPaths(PROJECT_ROOT)
paths.ensure_output_dirs

print("Project root:", PROJECT_ROOT)
print("Imported load_data successfully")

print("Imported libraries sucessfully! yay")

Project root: c:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_1_SupervisedLearning
Imported load_data successfully
Imported libraries sucessfully! yay


In [2]:
prices_raw, stock_list_raw, financials_raw = load_data()


Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/stock_prices.csv')]
Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/stock_list.csv')]
Matched files:  [WindowsPath('C:/Users/MY NGOC/Documents/MAU_TACAS_68803/DA631E_ArtificialIntelligenceForDataScience/Project_1_SupervisedLearning/data/train/financials.csv')]
prices: 0 rows with missing SecuritiesCode
stock_list: 0 rows with missing SecuritiesCode
financials: 2 rows with missing SecuritiesCode


In [3]:
prices_raw['SecuritiesCode'].nunique

<bound method IndexOpsMixin.nunique of 0          1301
1          1332
2          1333
3          1376
4          1377
           ... 
2332526    9990
2332527    9991
2332528    9993
2332529    9994
2332530    9997
Name: SecuritiesCode, Length: 2332531, dtype: string>

In [4]:
stock_list_raw[stock_list_raw['SecuritiesCode'] == '9997']

,SecuritiesCode,EffectiveDate,Name,Section/Products,NewMarketSegment,33SectorCode,33SectorName,17SectorCode,17SectorName,NewIndexSeriesSizeCode,NewIndexSeriesSize,TradeDate,Close,IssuedShares,MarketCapitalization,Universe0
4415,9997,1970-01-01 00:00:00.020211230,"BELLUNA CO.,LTD.",First Section (Domestic),Prime Market,6100,Retail Trade,14,RETAIL TRADE,6,TOPIX Small 1,1970-01-01 00:00:00.020211230,709.0,97244472.0,6.894633e+10,True


In [5]:
prices_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2332531 entries, 0 to 2332530
Data columns (total 12 columns):
 #   Column            Dtype         
---  ------            -----         
 0   RowId             object        
 1   Date              datetime64[ns]
 2   SecuritiesCode    string        
 3   Open              float64       
 4   High              float64       
 5   Low               float64       
 6   Close             float64       
 7   Volume            int64         
 8   AdjustmentFactor  float64       
 9   ExpectedDividend  float64       
 10  SupervisionFlag   bool          
 11  Target            float64       
dtypes: bool(1), datetime64[ns](1), float64(7), int64(1), object(1), string(1)
memory usage: 198.0+ MB


In [6]:

prices, prices_audit = clean_prices(df=prices_raw)
display(prices_audit)

assert len(prices_raw) == len(prices)
assert not prices.duplicated(['Date','SecuritiesCode']).any()

Fill missing adjustment with 1.0 (unchanged stock split)


{'input_rows': 2332531,
 'duplicates_stock_dates': 0,
 'missing_close': 7608,
 'missing_volume': 0,
 'negative_volume': 0,
 'invalid_ohcl': 0,
 'no_trade_rows': 7608,
 'partial_ohlc_rows': 0,
 'invalid_volume_rows': 0,
 'missing_adjustment': 0,
 'output_rows': 2332531,
 'removed_rows': 0}

In [7]:
no_trade_summary = pd.Series({
    "no_trade_rows": int(prices["NoTradeFlag"].sum()),
    "market_wide_no_trade_rows": int(prices["MarketWideNoTradeFlag"].sum()),
    "stock_specific_no_trade_rows": int(prices["StockSpecificNoTradeFlag"].sum()),
    "no_trade_rows_with_target": int(
        prices.loc[prices["NoTradeFlag"].eq(1), "Target"].notna().sum()
    ),
    "missing_target_rows_retained": int(prices["Target"].isna().sum()),
}, name="count")
display(no_trade_summary)

display(
    prices.loc[
        prices["NoTradeFlag"].eq(1),
        ["Date", "SecuritiesCode", "Open", "Close", "Volume", "Target",
         "NoTradeFlag", "MarketWideNoTradeFlag", "DaysSinceLastTrade"]
    ].head(10)
)

no_trade_rows                   7608
market_wide_no_trade_rows       1988
stock_specific_no_trade_rows    5620
no_trade_rows_with_target       7370
missing_target_rows_retained     238
Name: count, dtype: int64

,Date,SecuritiesCode,Open,Close,Volume,Target,NoTradeFlag,MarketWideNoTradeFlag,DaysSinceLastTrade
913,2020-10-01,1301,NaN,NaN,0,0.029208,1,1,1.0
2115,2020-10-01,1332,NaN,NaN,0,0.027211,1,1,1.0
3317,2020-10-01,1333,NaN,NaN,0,0.027695,1,1,1.0
3614,2020-10-01,1375,NaN,NaN,0,0.023833,1,1,1.0
4816,2020-10-01,1376,NaN,NaN,0,0.022152,1,1,1.0
6018,2020-10-01,1377,NaN,NaN,0,-0.006579,1,1,1.0
7220,2020-10-01,1379,NaN,NaN,0,0.003587,1,1,1.0
7728,2017-11-21,1381,NaN,NaN,0,0.013514,1,0,1.0
8372,2020-07-16,1381,NaN,NaN,0,0.004728,1,0,1.0
8422,2020-10-01,1381,NaN,NaN,0,-0.016248,1,1,1.0


In [8]:
assert len(prices) == len(prices_raw)

assert prices["NoTradeFlag"].sum() == 7608

assert not (
    prices["MarketWideNoTradeFlag"].eq(1)
    & prices["StockSpecificNoTradeFlag"].eq(1)
).any()

assert prices.loc[
    prices["StockSpecificNoTradeFlag"].eq(1),
    "NoTradeFlag"
].eq(1).all()

assert prices.loc[
    prices["Target"].isna(),
    "NoTradeFlag"
].eq(1).all()

print("All price-cleaning checks passed.")

All price-cleaning checks passed.


In [9]:
market_wide_dates = (
    prices.loc[
        prices["MarketWideNoTradeFlag"].eq(1)
    ]
    .groupby("Date")
    .agg(
        rows=("SecuritiesCode", "size"),
        no_trade_rows=("NoTradeFlag", "sum"),
        labeled_rows=("Target", "count")
    )
)

display(market_wide_dates)

,rows,no_trade_rows,labeled_rows
Date,,,
2020-10-01,1988,1988,1988


In [10]:
prices.isna().sum()

RowId                          0
Date                           0
SecuritiesCode                 0
Open                        7608
High                        7608
Low                         7608
Close                       7608
Volume                         0
AdjustmentFactor               0
ExpectedDividend               0
SupervisionFlag                0
Target                       238
PartialOHLCFlag                0
NoTradeFlag                    0
MarketWideNoTradeFlag          0
StockSpecificNoTradeFlag       0
AdjustmentEventFlag            0
HasExpectedDividend            0
LastTradeDate                264
DaysSinceLastTrade           264
CloseForFeatures             264
dtype: int64

In [11]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2332531 entries, 0 to 2332530
Data columns (total 21 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   RowId                     object        
 1   Date                      datetime64[ns]
 2   SecuritiesCode            string        
 3   Open                      float64       
 4   High                      float64       
 5   Low                       float64       
 6   Close                     float64       
 7   Volume                    int64         
 8   AdjustmentFactor          float64       
 9   ExpectedDividend          float64       
 10  SupervisionFlag           int8          
 11  Target                    float64       
 12  PartialOHLCFlag           int8          
 13  NoTradeFlag               int8          
 14  MarketWideNoTradeFlag     int8          
 15  StockSpecificNoTradeFlag  int8          
 16  AdjustmentEventFlag       int64         
 17  HasExpec

In [12]:
prices.describe()

,Date,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target,PartialOHLCFlag,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,AdjustmentEventFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CloseForFeatures
count,2332531,2.324923e+06,2.324923e+06,2.324923e+06,2.324923e+06,2.332531e+06,2.332531e+06,2.332531e+06,2.332531e+06,2.332293e+06,2332531.0,2.332531e+06,2.332531e+06,2.332531e+06,2.332531e+06,2.332531e+06,2332267,2.332267e+06,2.332267e+06
mean,2019-06-29 21:40:25.441719552,2.594511e+03,2.626540e+03,2.561227e+03,2.594023e+03,6.919366e+05,1.000508e+00,1.780746e-01,2.272210e-04,4.450962e-04,0.0,3.261693e-03,8.522931e-04,2.409400e-03,3.129648e-04,8.087781e-03,2019-06-29 23:18:18.354347776,5.972301e-03,2.600156e+03
min,2017-01-04 00:00:00,1.400000e+01,1.500000e+01,1.300000e+01,1.400000e+01,0.000000e+00,1.000000e-01,0.000000e+00,0.000000e+00,-5.785414e-01,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2017-01-04 00:00:00,0.000000e+00,1.400000e+01
25%,2018-04-05 00:00:00,1.022000e+03,1.035000e+03,1.009000e+03,1.022000e+03,3.030000e+04,1.000000e+00,0.000000e+00,0.000000e+00,-1.049869e-02,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2018-04-06 00:00:00,0.000000e+00,1.022000e+03
50%,2019-07-05 00:00:00,1.812000e+03,1.834000e+03,1.790000e+03,1.811000e+03,1.071000e+05,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2019-07-05 00:00:00,0.000000e+00,1.813000e+03
75%,2020-09-28 00:00:00,3.030000e+03,3.070000e+03,2.995000e+03,3.030000e+03,4.021000e+05,1.000000e+00,0.000000e+00,0.000000e+00,1.053159e-02,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2020-09-28 00:00:00,0.000000e+00,3.035000e+03
max,2021-12-03 00:00:00,1.099500e+05,1.105000e+05,1.072000e+05,1.095500e+05,6.436540e+08,2.000000e+01,1.070000e+03,1.000000e+00,1.119512e+00,0.0,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2021-12-03 00:00:00,2.700000e+01,1.095500e+05
std,NaN,3.577192e+03,3.619363e+03,3.533494e+03,3.576538e+03,3.911256e+06,6.773040e-02,3.333284e+00,1.507214e-02,2.339879e-02,0.0,5.701803e-02,2.918162e-02,4.902648e-02,1.768805e-02,8.956770e-02,NaN,1.465763e-01,3.593742e+03


## Stock list

In [13]:
stock_list, stock_list_audit = clean_stock_list(df=stock_list_raw)

In [14]:
stock_list.head()

,SecuritiesCode,33SectorCode,17SectorCode,NewMarketSegment,NewIndexSeriesSizeCode,IssuedShares,MarketCapitalization,Universe0,EffectiveDate,TradeDate,IssuedSharesLog,MarketCapitalizationLog
0,1301,50,1,Prime Market,7,1.092828e+07,3.365911e+10,True,2021-12-30,2021-12-30,16.206865,24.239550
1,1305,__MISSING__,__MISSING__,__MISSING__,__MISSING__,3.634636e+09,7.621831e+12,False,2021-12-30,2021-12-30,22.013775,29.662038
2,1306,__MISSING__,__MISSING__,__MISSING__,__MISSING__,7.917718e+09,1.641739e+13,False,2021-12-30,2021-12-30,22.792369,30.429362
3,1308,__MISSING__,__MISSING__,__MISSING__,__MISSING__,3.736943e+09,7.671945e+12,False,2021-12-30,2021-12-30,22.041534,29.668591
4,1309,__MISSING__,__MISSING__,__MISSING__,__MISSING__,7.263200e+04,3.216145e+09,False,2021-12-30,2021-12-30,11.193175,21.891449


In [15]:
stock_list.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4417 entries, 0 to 4416
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   SecuritiesCode           4417 non-null   string        
 1   33SectorCode             4417 non-null   string        
 2   17SectorCode             4417 non-null   string        
 3   NewMarketSegment         4417 non-null   string        
 4   NewIndexSeriesSizeCode   4417 non-null   string        
 5   IssuedShares             4121 non-null   float64       
 6   MarketCapitalization     4121 non-null   float64       
 7   Universe0                4417 non-null   bool          
 8   EffectiveDate            4417 non-null   datetime64[ns]
 9   TradeDate                4121 non-null   datetime64[ns]
 10  IssuedSharesLog          4121 non-null   float64       
 11  MarketCapitalizationLog  4121 non-null   float64       
dtypes: bool(1), datetime64[ns](2), flo

#### Verify that every SecuritiesCode in prices has stock_list metadata


In [16]:
prices_codes = set(prices['SecuritiesCode'])
stock_list_codes = set(stock_list['SecuritiesCode'])

missing_stock_list_codes = prices_codes - stock_list_codes

print("Prices's SecuritiesCodes without metadata in stock_list: ", len(missing_stock_list_codes))
print(missing_stock_list_codes)

Prices's SecuritiesCodes without metadata in stock_list:  0
set()


In [17]:
stock_list[stock_list['SecuritiesCode'] == '9997']

,SecuritiesCode,33SectorCode,17SectorCode,NewMarketSegment,NewIndexSeriesSizeCode,IssuedShares,MarketCapitalization,Universe0,EffectiveDate,TradeDate,IssuedSharesLog,MarketCapitalizationLog
4415,9997,6100,14,Prime Market,6,97244472.0,6.894633e+10,True,2021-12-30,2021-12-30,18.392739,24.956594


In [18]:
prices[prices['SecuritiesCode'] == '9997']

,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,...,Target,PartialOHLCFlag,NoTradeFlag,MarketWideNoTradeFlag,StockSpecificNoTradeFlag,AdjustmentEventFlag,HasExpectedDividend,LastTradeDate,DaysSinceLastTrade,CloseForFeatures
2331329,20170104_9997,2017-01-04,9997,729.0,800.0,729.0,778.0,359200,1.0,0.0,...,0.007481,0,0,0,0,0,0,2017-01-04,0.0,778.0
2331330,20170105_9997,2017-01-05,9997,793.0,809.0,780.0,802.0,264200,1.0,0.0,...,-0.004950,0,0,0,0,0,0,2017-01-05,0.0,802.0
2331331,20170106_9997,2017-01-06,9997,798.0,809.0,792.0,808.0,162900,1.0,0.0,...,-0.006219,0,0,0,0,0,0,2017-01-06,0.0,808.0
2331332,20170110_9997,2017-01-10,9997,804.0,815.0,796.0,804.0,168000,1.0,0.0,...,-0.028786,0,0,0,0,0,0,2017-01-10,0.0,804.0
2331333,20170111_9997,2017-01-11,9997,800.0,805.0,795.0,799.0,151400,1.0,0.0,...,0.009021,0,0,0,0,0,0,2017-01-11,0.0,799.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2332526,20211129_9997,2021-11-29,9997,678.0,679.0,665.0,668.0,320800,1.0,0.0,...,0.026987,0,0,0,0,0,0,2021-11-29,0.0,668.0
2332527,20211130_9997,2021-11-30,9997,670.0,689.0,667.0,667.0,296300,1.0,0.0,...,-0.001460,0,0,0,0,0,0,2021-11-30,0.0,667.0
2332528,20211201_9997,2021-12-01,9997,661.0,688.0,660.0,685.0,339100,1.0,0.0,...,0.017544,0,0,0,0,0,0,2021-12-01,0.0,685.0
2332529,20211202_9997,2021-12-02,9997,681.0,692.0,680.0,684.0,342900,1.0,0.0,...,0.014368,0,0,0,0,0,0,2021-12-02,0.0,684.0


### financials

In [19]:
financials_raw.head()

,DisclosureNumber,DateCode,Date,SecuritiesCode,DisclosedDate,DisclosedTime,DisclosedUnixTime,TypeOfDocument,CurrentPeriodEndDate,TypeOfCurrentPeriod,...,ForecastEarningsPerShare,ApplyingOfSpecificAccountingOfTheQuarterlyFinancialStatements,MaterialChangesInSubsidiaries,ChangesBasedOnRevisionsOfAccountingStandard,ChangesOtherThanOnesBasedOnRevisionsOfAccountingStandard,ChangesInAccountingEstimates,RetrospectiveRestatement,NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock,NumberOfTreasuryStockAtTheEndOfFiscalYear,AverageNumberOfShares
0,2.016121e+13,20170104_2753,2017-01-04,2753,2017-01-04,07:30:00,1.483483e+09,3QFinancialStatements_Consolidated_JP,2016-12-31,3Q,...,319.76,NaN,False,True,False,False,False,6848800,－,6848800
1,2.017010e+13,20170104_3353,2017-01-04,3353,2017-01-04,15:00:00,1.483510e+09,3QFinancialStatements_Consolidated_JP,2016-11-30,3Q,...,485.36,NaN,False,True,False,False,False,2035000,118917,1916083
2,2.016123e+13,20170104_4575,2017-01-04,4575,2017-01-04,12:00:00,1.483499e+09,ForecastRevision,2016-12-31,2Q,...,-93.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.017010e+13,20170105_2659,2017-01-05,2659,2017-01-05,15:00:00,1.483596e+09,3QFinancialStatements_Consolidated_JP,2016-11-30,3Q,...,285.05,NaN,False,True,False,False,False,31981654,18257,31963405
4,2.017011e+13,20170105_3050,2017-01-05,3050,2017-01-05,15:30:00,1.483598e+09,ForecastRevision,2017-02-28,FY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
financials_raw.columns

Index(['DisclosureNumber', 'DateCode', 'Date', 'SecuritiesCode',
       'DisclosedDate', 'DisclosedTime', 'DisclosedUnixTime', 'TypeOfDocument',
       'CurrentPeriodEndDate', 'TypeOfCurrentPeriod',
       'CurrentFiscalYearStartDate', 'CurrentFiscalYearEndDate', 'NetSales',
       'OperatingProfit', 'OrdinaryProfit', 'Profit', 'EarningsPerShare',
       'TotalAssets', 'Equity', 'EquityToAssetRatio', 'BookValuePerShare',
       'ResultDividendPerShare1stQuarter', 'ResultDividendPerShare2ndQuarter',
       'ResultDividendPerShare3rdQuarter',
       'ResultDividendPerShareFiscalYearEnd', 'ResultDividendPerShareAnnual',
       'ForecastDividendPerShare1stQuarter',
       'ForecastDividendPerShare2ndQuarter',
       'ForecastDividendPerShare3rdQuarter',
       'ForecastDividendPerShareFiscalYearEnd',
       'ForecastDividendPerShareAnnual', 'ForecastNetSales',
       'ForecastOperatingProfit', 'ForecastOrdinaryProfit', 'ForecastProfit',
       'ForecastEarningsPerShare',
       'ApplyingOf

In [21]:
financials, financials_audit = clean_financials(df=financials_raw)

In [22]:
display(financials_audit)
display(financials.head())



input_rows                      92956
rows_with_valid_key_and_date    87293
output_rows                     87293
numeric_source_columns             16
availability_lag_days               1
dtype: int64

,SecuritiesCode,DisclosedDate,FinancialAvailableDate,Fin_OperatingMargin,Fin_OrdinaryMargin,Fin_ProfitMargin,Fin_ReturnOnAssets,Fin_NetSales_SignedLog,Fin_OperatingProfit_SignedLog,Fin_OrdinaryProfit_SignedLog,...,Fin_Equity_SignedLog,Fin_EquityToAssetRatio_SignedLog,Fin_BookValuePerShare_SignedLog,Fin_ResultDividendPerShareAnnual_SignedLog,Fin_ForecastDividendPerShareAnnual_SignedLog,Fin_ForecastNetSales_SignedLog,Fin_ForecastOperatingProfit_SignedLog,Fin_ForecastOrdinaryProfit_SignedLog,Fin_ForecastProfit_SignedLog,Fin_ForecastEarningsPerShare_SignedLog
0,2753,2017-01-04,2017-01-05,0.094328,0.098150,0.065639,0.817252,23.848314,21.487337,21.527060,...,23.629894,0.597187,7.890740,NaN,4.615121,24.182732,21.903458,21.917188,21.507167,5.770693
1,3353,2017-01-04,2017-01-05,0.037057,0.035159,0.028426,0.301434,23.820110,20.524815,20.472237,...,22.746930,0.263133,NaN,NaN,4.290459,24.131108,21.023370,20.985630,20.650695,6.186949
2,4575,2017-01-04,2017-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,18.515991,-19.957548,-19.959696,-19.961840,-4.544464
3,2659,2017-01-05,2017-01-06,0.083454,0.085754,0.053205,0.785473,25.626917,23.143456,23.170644,...,25.337445,0.568151,8.030774,NaN,3.761200,25.903267,23.374252,23.395551,22.932748,5.656167
4,3050,2017-01-05,2017-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.218876,NaN,NaN,NaN,NaN,NaN


In [23]:
assert not financials.duplicated(['SecuritiesCode', 'FinancialAvailableDate']).any()

assert (financials['FinancialAvailableDate'] >= financials['DisclosedDate']).all()

In [27]:
from operator import index


interim_paths = {
    'prices_clean': paths.interim / "stock_prices_clean.parquet"
    ,'stock_list_clean': paths.interim / "stock_list_clean.parquet"
    ,'financials_clean': paths.interim / "financials_clean.parquet"
}
prices.to_parquet(interim_paths['prices_clean'],index=False)
stock_list.to_parquet(interim_paths['stock_list_clean'],index=False)
financials.to_parquet(interim_paths['financials_clean'],index=False)

pd.DataFrame({
    "artifact": interim_paths.keys(),
    "path": [str(path.relative_to(PROJECT_ROOT)) for path in interim_paths.values()],
    "exists": [path.is_file() for path in interim_paths.values()],
})

,artifact,path,exists
0,prices_clean,data\interim\stock_prices_clean.parquet,True
1,stock_list_clean,data\interim\stock_list_clean.parquet,True
2,financials_clean,data\interim\financials_clean.parquet,True
